# Personalized Learning Support Recommendation Engine

This notebook inspects recommendations generated from the saved Random Forest prediction, early weeks 1-6 student-topic signals, and per-student SHAP contributions. It provides supportive learning guidance rather than labels or punitive decisions.

Future weeks 7-10 topic performance and the future-derived topic target are loaded only for validation/reference. They are not used to generate recommendation priorities or actions. SHAP contributions describe model behavior and do not establish causation.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from recommendations import generate_student_support, _load_inputs, _thresholds

early, topic, topic_summary, model, feature_columns, test_ids = _load_inputs()
recommendation_path = PROJECT_ROOT / 'models' / 'recommendation_examples.csv'
recommendations = pd.read_csv(recommendation_path)
print('Students:', early['student_id'].nunique())
print('Topics analyzed:', topic['topic'].nunique())
print('Recommendation rows:', len(recommendations))
print('Model:', type(model).__name__)
print('Features used:', len(feature_columns))

## 1. Transparent Early-Signal Rules

In [ ]:
thresholds = _thresholds(topic)
display(pd.Series(thresholds, name='25th_percentile_threshold').to_frame())
print('Points: low early score=3; declining trend=2; low question accuracy=2; low practice activity=1')
print('Priority: High >=5; Medium 3-4; Support 1-2; Monitor 0')
print('All topic concepts come from the actual dataset topic names.')

## 2. Recommendation Coverage and Priority

In [ ]:
print('Predicted learning status counts:')
display(early['academic_difficulty'].value_counts().sort_index().rename(index={0: 'Low', 1: 'Moderate', 2: 'High'}).to_frame('students'))
print('Recommendation priority distribution:')
display(recommendations['priority'].value_counts().to_frame('recommendation_rows'))
print('Most frequently recommended topics:')
display(recommendations['topic'].value_counts().head(15).to_frame('recommendation_count'))

## 3. Example Personalized Reports

In [ ]:
example_rows = []
for student_id in test_ids[:5]:
    report = generate_student_support(student_id)
    for topic in report['topics_needing_attention'][:3]:
        example_rows.append({
            'student_id': student_id,
            'learning_status': report['predicted_learning_status'],
            'topic': topic['topic'],
            'priority': topic['priority'],
            'reason': '; '.join(topic['reasons']),
            'recommended_support': ' '.join(topic['recommended_actions']),
        })
display(pd.DataFrame(example_rows))
print('Reports are also saved under models/recommendation_examples/.')

## 4. Leakage and Scope Checks

- Recommendations use only early weeks 1-6 engineered topic features and the student-level model's early feature matrix.
- `student_id` is an identifier used for lookup, not a predictive feature.
- `academic_difficulty`, future topic scores, and future topic status are not inputs to recommendation priority or action.
- SHAP reasons correspond to the saved model feature list.
- Every recommended topic is one of the 25 topics present in the dataset.
- The existing topic target and summary are reference artifacts; using them to make early recommendations would leak future information, so they are not used for the rule scores.

The recommendation engine supports learning improvement conversations and does not guarantee improvement or establish causation.